# 2.2. Modelos Lineales

## Setup

In [71]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import numpy as np
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate, KFold
from sklearn.metrics import make_scorer, accuracy_score, recall_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

import pandas as pd
import numpy as np
import plotly.io as pio
import plotly.figure_factory as ff
import warnings

pio.templates.default = "plotly_dark"
warnings.simplefilter(action='ignore', category=FutureWarning)


In [2]:
session_info.show()

In [9]:
# Funciones
def color_negative_red(val,value):
    if val == 1:
        color = 'grey'
    elif np.abs(val) > value:
        color = 'red'
    else:
        color = 'white'
    return f'color: {color}'
def get_style(data, fun, **kw):
    styled_data = data.style.applymap(fun, **kw)
    styled_data.to_html('styled_data.html', escape=False)
    return styled_data
def get_cv_scores_report_classification(estimator, X, y, n_splits):
    cv_scores = cross_validate(
                    estimator = estimator,
                    X         = X,
                    y         = y,
                    scoring   = {
                                'accuracy': make_scorer(accuracy_score), 
                                'recall': make_scorer(recall_score, average='weighted'), 
                                'roc_auc_ovr': 'roc_auc_ovr'
                                },
                    cv        = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=5, random_state=333),
                )
    
    # Convertir el diccionario a dataframe para facilitar la visualización
    cv_scores = pd.DataFrame(cv_scores)
    print(f"Accuracy en CV: mean {cv_scores.test_accuracy.mean():.2f}, std {cv_scores.test_accuracy.std():.2f}")
    print(f"Recall en CV: mean {cv_scores.test_recall.mean():.2f}, std {cv_scores.test_recall.std():.2f}")
    print(f"ROC AUC en CV: mean {cv_scores.test_roc_auc_ovr.mean():.2f}, std {cv_scores.test_roc_auc_ovr.std():.2f}")

## 2.2.4. Modelos Regularizados

### Contexto

La función de costo de la regresión logística (sin regularización) es:

$$ J(\beta) = -\frac{1}{n} \sum_{i=1}^{n} [y^{(i)} \log(f_{\beta}(x^{(i)})) + (1 - y^{(i)}) \log(1 - f_{\beta}(x^{(i)}))] $$

Donde:

- $ n $ es tamaño de muestra.
- $ f_{\beta}(x) $ es la función sigmoide y está definida como $ f_{\beta}(x) = \frac{1}{1 + e^{-\beta^T x}} $.
- $ y^{(i)} $ es la etiqueta verdadera del $i$-ésimo ejemplo.
- $ x^{(i)} $ es el vector de características del $i$-ésimo ejemplo.
- $ \beta $ es el vector de coeficientes (o parámetros) del modelo.

Para regularizar la regresión logística, podemos añadir una penalización a la función de costo. La penalización es típicamente o la norma L1 (para regularización Lasso) o la norma L2 (para regularización Ridge) de los coeficientes.

1. **Regularización L1 (Lasso)**:

$$ J(\beta) = -\frac{1}{n} \sum_{i=1}^{n} [y^{(i)} \log(f_{\beta}(x^{(i)})) + (1 - y^{(i)}) \log(1 - f_{\beta}(x^{(i)}))] + \lambda \sum_{j=1}^{n} |\beta_j| $$

2. **Regularización L2 (Ridge)**:

$$ J(\beta) = -\frac{1}{n} \sum_{i=1}^{n} [y^{(i)} \log(f_{\beta}(x^{(i)})) + (1 - y^{(i)}) \log(1 - f_{\beta}(x^{(i)}))] + \frac{\lambda}{2} \sum_{j=1}^{n} \beta_j^2 $$

Donde:

- $ n $ es el número de características (incluyendo el término de intercepción).
- $ \lambda $ es el parámetro de regularización. Un valor más alto de $ \lambda $ significa más regularización.<span style="color:red">R1</span>

### Ejemplo

Para brevemente este modelo utilizaremos el conjunto de datos `Breast Cancer Wisconsin` la cuál refiere en conjunto de datos atemporales que contiene información de 400 clientes. Nuestra tarea consistirá en predecir los ingresos a partir de nuestras características.

| Característica                  | Descripción                                                                       |
|---------------------------------|-----------------------------------------------------------------------------------|
| **Muestra**                     | 569                                                                               |
| **Número de Atributos**         | 30 numéricos, atributos predictivos y la clase                                    |
| **Atributos**    | - radio (media de distancias desde el centro a los puntos del perímetro)          |
|                                 | - textura (desviación estándar de valores en escala de grises)                    |
|                                 | - perímetro                                                                      |
|                                 | - área                                                                            |
|                                 | - suavidad (variación local en longitudes de radio)                               |
|                                 | - compactación (perímetro^2 / área - 1.0)                                        |
|                                 | - concavidad (gravedad de las porciones cóncavas del contorno)                    |
|                                 | - puntos cóncavos (número de porciones cóncavas del contorno)                    |
|                                 | - simetría                                                                        |
|                                 | - dimensión fractal ("aproximación de la línea de costa" - 1)                     |
| **Distribución de Clases**      | 212 - Maligno, 357 - Benigno                                                      |


In [23]:
data = datasets.load_breast_cancer()
X = pd.DataFrame(data.data,columns= data['feature_names'])
y = pd.Series(data.target)

In [35]:
y.value_counts()

1    357
0    212
Name: count, dtype: int64

In [32]:
get_style(X.corr(), color_negative_red, value=0.4)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
mean radius,1.000000,0.323782,0.997855,0.987357,0.170581,0.506124,0.676764,0.822529,0.147741,-0.311631,0.679090,-0.097317,0.674172,0.735864,-0.222600,0.206000,0.194204,0.376169,-0.104321,-0.042641,0.969539,0.297008,0.965137,0.941082,0.119616,0.413463,0.526911,0.744214,0.163953,0.007066
mean texture,0.323782,1.000000,0.329533,0.321086,-0.023389,0.236702,0.302418,0.293464,0.071401,-0.076437,0.275869,0.386358,0.281673,0.259845,0.006614,0.191975,0.143293,0.163851,0.009127,0.054458,0.352573,0.912045,0.358040,0.343546,0.077503,0.277830,0.301025,0.295316,0.105008,0.119205
mean perimeter,0.997855,0.329533,1.000000,0.986507,0.207278,0.556936,0.716136,0.850977,0.183027,-0.261477,0.691765,-0.086761,0.693135,0.744983,-0.202694,0.250744,0.228082,0.407217,-0.081629,-0.005523,0.969476,0.303038,0.970387,0.941550,0.150549,0.455774,0.563879,0.771241,0.189115,0.051019
mean area,0.987357,0.321086,0.986507,1.000000,0.177028,0.498502,0.685983,0.823269,0.151293,-0.283110,0.732562,-0.066280,0.726628,0.800086,-0.166777,0.212583,0.207660,0.372320,-0.072497,-0.019887,0.962746,0.287489,0.959120,0.959213,0.123523,0.390410,0.512606,0.722017,0.143570,0.003738
mean smoothness,0.170581,-0.023389,0.207278,0.177028,1.000000,0.659123,0.521984,0.553695,0.557775,0.584792,0.301467,0.068406,0.296092,0.246552,0.332375,0.318943,0.248396,0.380676,0.200774,0.283607,0.213120,0.036072,0.238853,0.206718,0.805324,0.472468,0.434926,0.503053,0.394309,0.499316
mean compactness,0.506124,0.236702,0.556936,0.498502,0.659123,1.000000,0.883121,0.831135,0.602641,0.565369,0.497473,0.046205,0.548905,0.455653,0.135299,0.738722,0.570517,0.642262,0.229977,0.507318,0.535315,0.248133,0.590210,0.509604,0.565541,0.865809,0.816275,0.815573,0.510223,0.687382
mean concavity,0.676764,0.302418,0.716136,0.685983,0.521984,0.883121,1.000000,0.921391,0.500667,0.336783,0.631925,0.076218,0.660391,0.617427,0.098564,0.670279,0.691270,0.683260,0.178009,0.449301,0.688236,0.299879,0.729565,0.675987,0.448822,0.754968,0.884103,0.861323,0.409464,0.514930
mean concave points,0.822529,0.293464,0.850977,0.823269,0.553695,0.831135,0.921391,1.000000,0.462497,0.166917,0.698050,0.021480,0.710650,0.690299,0.027653,0.490424,0.439167,0.615634,0.095351,0.257584,0.830318,0.292752,0.855923,0.809630,0.452753,0.667454,0.752399,0.910155,0.375744,0.368661
mean symmetry,0.147741,0.071401,0.183027,0.151293,0.557775,0.602641,0.500667,0.462497,1.000000,0.479921,0.303379,0.128053,0.313893,0.223970,0.187321,0.421659,0.342627,0.393298,0.449137,0.331786,0.185728,0.090651,0.219169,0.177193,0.426675,0.473200,0.433721,0.430297,0.699826,0.438413
mean fractal dimension,-0.311631,-0.076437,-0.261477,-0.283110,0.584792,0.565369,0.336783,0.166917,0.479921,1.000000,0.000111,0.164174,0.039830,-0.090170,0.401964,0.559837,0.446630,0.341198,0.345007,0.688132,-0.253691,-0.051269,-0.205151,-0.231854,0.504942,0.458798,0.346234,0.175325,0.334019,0.767297


In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [52]:
# Premodelado: 
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Definir la función objetivo para hyperopt
def objective(params):
    model = LogisticRegression(**params, max_iter=10000, solver='saga')
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = f1_score(y_test, predictions)
    return {'loss': -accuracy, 'status': STATUS_OK}

# Espacio hiperparametral
space = {
    'penalty': hp.choice('penalty', ['l1', 'l2']),
    'C': hp.loguniform('C', np.log(0.001), np.log(10))
}

trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials
)

100%|██████████| 100/100 [00:07<00:00, 13.08trial/s, best loss: -0.9861111111111112]


In [55]:
lrr =LogisticRegression(C = best['C'],
                        penalty= 'l' + str(best['penalty']), 
                        max_iter=10000, 
                        solver='saga')
lrr.fit(X_train,y_train)

LogisticRegression(C=0.2115994772513103, max_iter=10000, penalty='l1',
                   solver='saga')

In [66]:
#y_test_pred = lrr.predict(X_test)

get_cv_scores_report_classification(lrr,X_test,y_test,4)

Accuracy en CV: mean 0.94, std 0.03
Recall en CV: mean 0.94, std 0.03
ROC AUC en CV: mean 1.00, std 0.00


In [72]:
# Realizar predicciones
y_pred = lrr.predict(X_test)

# Crear una matriz de confusión
cm = confusion_matrix(y_test, y_pred)
ff.create_annotated_heatmap(z=cm).update_layout(title='Matriz de Confusión', xaxis_title='Clases Predichas', yaxis_title='Clases Actuales')